# Project-Synonymtest mit Azure CLI und REST


## 1. Daten

* [Dauerhafte Project-Entity mit qwert](../ontology/entities/Project.tmdl)
* [Generierte Kopie der dauerhaften Project-Entity](fixtures/project_synonym/Project.tmdl)

In [1]:
if [[ -f ../ontology/entities/Project.tmdl ]]; then cd ..; fi
RG=rg-projplan-demo
AGENT=project-planning-agent
QUESTION="Gib mir eine Liste aller qwert mit ihrer ID und ihrem Namen."
FABRIC=https://api.fabric.microsoft.com/v1
WORKSPACE_NAME=ProjectPlanning
SEMANTIC_MODEL_NAME=ProjectPlanningModel
ONTOLOGY_NAME=$(jq -r '.metadata.displayName' ontology/.platform)
PROJECT_ENDPOINT=$(az deployment group show -g "$RG" -n main \
  --query properties.outputs.foundryProjectEndpoint.value -o tsv) &&
WORKSPACE_ID=$(az rest --url "$FABRIC/workspaces" --resource https://api.fabric.microsoft.com \
  --query "value[?displayName=='$WORKSPACE_NAME'].id | [0]" -o tsv) &&
ONTOLOGY_ID=$(az rest --url "$FABRIC/workspaces/$WORKSPACE_ID/ontologies" --resource https://api.fabric.microsoft.com \
  --query "value[?displayName=='$ONTOLOGY_NAME'].id | [0]" -o tsv) &&
SEMANTIC_MODEL_ID=$(az rest --url "$FABRIC/workspaces/$WORKSPACE_ID/semanticModels" --resource https://api.fabric.microsoft.com \
  --query "value[?displayName=='$SEMANTIC_MODEL_NAME'].id | [0]" -o tsv) &&
ONTOLOGY_URL="$FABRIC/workspaces/$WORKSPACE_ID/ontologies/$ONTOLOGY_ID" &&
[[ -n "$WORKSPACE_ID" && -n "$ONTOLOGY_ID" && -n "$SEMANTIC_MODEL_ID" ]] &&
printf '%s=%s\n' \
  WORKSPACE_NAME "$WORKSPACE_NAME" \
  SEMANTIC_MODEL_NAME "$SEMANTIC_MODEL_NAME" \
  ONTOLOGY_NAME "$ONTOLOGY_NAME" \
  WORKSPACE_ID "$WORKSPACE_ID" \
  ONTOLOGY_ID "$ONTOLOGY_ID" \
  SEMANTIC_MODEL_ID "$SEMANTIC_MODEL_ID" \
  ONTOLOGY_URL "$ONTOLOGY_URL"


WORKSPACE_NAME=ProjectPlanning


SEMANTIC_MODEL_NAME=ProjectPlanningModel


ONTOLOGY_NAME=ProjectPlanning


WORKSPACE_ID=66d9c2b2-d35e-4555-9e74-926164ce584c


ONTOLOGY_ID=8750de7f-d46f-4dd4-84f4-a7e31f68ff24


SEMANTIC_MODEL_ID=22fb9927-ac28-4092-80c0-0f3d7a56a156


ONTOLOGY_URL=https://api.fabric.microsoft.com/v1/workspaces/66d9c2b2-d35e-4555-9e74-926164ce584c/ontologies/8750de7f-d46f-4dd4-84f4-a7e31f68ff24


### Gespeicherte Agentenanweisungen bereitstellen

[update_agent_instructions.py](../scripts/update_agent_instructions.py) vergleicht
die Repository-Anweisungen mit der letzten veroeffentlichten Version.
Nur bei einer Aenderung wird eine neue Version desselben Agenten erstellt.
Modell, Tools und sonstige Definitionseinstellungen bleiben erhalten.
Alle folgenden Anfragen verwenden die ausgegebene Version explizit.


In [2]:
AGENT_VERSION=$(.venv/bin/python scripts/update_agent_instructions.py "$PROJECT_ENDPOINT" "$AGENT") &&
printf 'AGENT_VERSION=%s\n' "$AGENT_VERSION"


AGENT_VERSION=7


## 3. Vollstaendige Upload-Dateien erzeugen

* [build_synonym_definitions.py - das Build-Script](../scripts/build_synonym_definitions.py)
* [fabric-ontology-without-qwert.json - temporaere Negativkontrolle](fixtures/project_synonym/fabric-ontology-without-qwert.json)
* [fabric-ontology-repository.json - dauerhafte Definition mit qwert zum Wiederherstellen](fixtures/project_synonym/fabric-ontology-repository.json)
* [fabric-ontology-qwert.json - Kopie der Repository-Definition mit qwert](fixtures/project_synonym/fabric-ontology-qwert.json)

In [3]:
.venv/bin/python scripts/build_synonym_definitions.py "$WORKSPACE_ID" "$WORKSPACE_NAME" "$SEMANTIC_MODEL_ID" "$SEMANTIC_MODEL_NAME"


Created /home/ga1/cptdazagentprojectmanagment/notebooks/fixtures/project_synonym/fabric-ontology-repository.json


Created /home/ga1/cptdazagentprojectmanagment/notebooks/fixtures/project_synonym/fabric-ontology-qwert.json


Created /home/ga1/cptdazagentprojectmanagment/notebooks/fixtures/project_synonym/fabric-ontology-without-qwert.json


### Vollstaendige Upload-Dateien

* [fabric-ontology-without-qwert.json](fixtures/project_synonym/fabric-ontology-without-qwert.json)
* [fabric-ontology-repository.json](fixtures/project_synonym/fabric-ontology-repository.json)
* [fabric-ontology-qwert.json](fixtures/project_synonym/fabric-ontology-qwert.json)


### Project-TMDL aus beiden Upload-Dateien vergleichen

* [REST-Endpunkt Fabric Ontology Update](https://learn.microsoft.com/rest/api/fabric/ontology/items/update-ontology-definition#payloadtype)
* [Alternative via Git-Integration](https://learn.microsoft.com/rest/api/fabric/articles/item-management/definitions/ontology-definition)

In [4]:
(
  set -euo pipefail
  for file in notebooks/fixtures/project_synonym/fabric-ontology-{without-qwert,repository}.json; do
    printf '\n== %s\n' "${file##*/}"
    jq -er '
      [.definition.parts[] | select(.path=="entities/Project.tmdl" and .payloadType=="InlineBase64")]
      | if length==1 then .[0].payload | @base64d
        else error("Genau ein Project-TMDL erwartet") end
    ' "$file" | sed -n '1,12p'
  done
)


== fabric-ontology-without-qwert.json


/// A construction project for a customer.


entity Project


	lineageTag: 762fca16-9cf7-58f4-b0ad-4726a0654551


	backingTable: projects


	keyProperty: ProjectId


... (14 outputs left out)

	synonym qwert


	/// Unique project ID, for example P-100.


	property ProjectId


		dataType: string


### Definierten Ausgangszustand ohne qwert herstellen

Der Repository-Stand enthaelt qwert dauerhaft. Nur fuer die Negativkontrolle
spielt diese Zelle eine abgeleitete Definition ohne qwert ein.
Nur in Fabric vorhandene Aenderungen werden ueberschrieben.
Nach dem Test bleibt das explizite Ruecksetzen in Abschnitt 8 erforderlich,
auch nach Fehlern oder einem vorzeitigen Abbruch. Dabei wird qwert wiederhergestellt.


In [5]:
.venv/bin/python scripts/upload_ontology_definition.py "$WORKSPACE_ID" "$ONTOLOGY_ID" notebooks/fixtures/project_synonym/fabric-ontology-without-qwert.json


Ontology updated from notebooks/fixtures/project_synonym/fabric-ontology-without-qwert.json (HTTP 200).


In [6]:
(
  set -euo pipefail
  az rest --method post --url "$ONTOLOGY_URL/getDefinition" \
    --resource https://api.fabric.microsoft.com --body '{}' --output json |
    jq -er '
      def has_qwert: split("\n") | any(rtrimstr("\r") == "\tsynonym qwert");
      [.definition.parts[] | select(.path=="entities/Project.tmdl" and .payloadType=="InlineBase64")]
      | if length==1 then .[0].payload | @base64d
        else error("Genau ein Project-TMDL aus Fabric erwartet") end
      | if (has_qwert | not) then .
        else error("Unerwarteter qwert-Stand in Fabric") end
    ' |
    sed -n 's/\r$//; /^\/\/\//p; /^entity /p; /^\tsynonym /p; /^\tannotation /p'
)


/// A construction project for a customer.


entity Project


	synonym 'Construction project'


	synonym 'Building project'


## 4. Agentenanfragen vorbereiten

`REQUEST` enthaelt nur die normale Endnutzerfrage. Die Routing-Regeln liegen
in den gespeicherten Agentenanweisungen, nicht im Nutzerprompt.
`REQUEST_WITH_HINTS` bleibt als zusaetzliche Diagnose erhalten.
Beide Requests verwenden dieselbe explizite Agentenversion und `store:false`,
ohne `conversation` oder `previous_response_id`.

Alle JSON-Antworten werden direkt mit `jq .` formatiert, ohne Python-Ausgabehelfer.
`calls[].arguments` zeigt die ans Tool weitergeleitete Frage. Diese kann vom
urspruenglichen Nutzertext abweichen; die tatsaechlich gesendeten Requests sind
oben in den Vorbereitungszellen sichtbar.
[check_agent_response.jq](../scripts/check_agent_response.jq) prueft nach der
Ausgabe den Status und API-/Tool-Fehler, einschliesslich verschachtelter
Dienstfehler in `additionalProperties._meta`, ohne die Antwort zu veraendern.

Die serverseitige Meldung `Error during output moderation for tool: ask_ontology
for general harms.` bleibt in `calls[].result` unter `raiPostToolAnnotations`
sichtbar. Sie tritt auch bei normalen Projektfragen ohne `qwert` auf und ist
keine bestaetigte Inhaltsblockierung. Die zusaetzliche lokale Python-Warnung
wird nicht mehr erzeugt. Eine abgeschlossene Datenabfrage ist deshalb aber
noch kein Beleg fuer eine erfolgreich durchgefuehrte Moderation.


In [7]:
REQUEST=$(jq -n --arg agent "$AGENT" --arg version "$AGENT_VERSION" --arg question "$QUESTION" \
  '{agent_reference:{type:"agent_reference",name:$agent,version:$version},input:$question,store:false}') &&
printf '%s\n' "$REQUEST" | jq .


{


  "agent_reference": {


    "type": "agent_reference",


    "name": "project-planning-agent",


    "version": "7"


  },


  "input": "Gib mir eine Liste aller qwert mit ihrer ID und ihrem Namen.",


  "store": false


}


In [8]:
REQUEST_WITH_HINTS=$(jq -ne --argjson request "${REQUEST:-null}" \
  --arg hint "Nutze ask_ontology, um die tatsaechlichen Datensaetze abzufragen, nicht nur Entity-Metadaten. Falls der Begriff unbekannt ist, sage dies, statt eine Bedeutung zu erraten." \
  '$request | if (.input | type) == "string" and (.input | length) > 0
    then .input += " " + $hint
    else error("REQUEST fehlt oder enthaelt keine Frage. Zuerst die REST-Anfrage vorbereiten.") end') &&
printf '%s\n' "$REQUEST_WITH_HINTS" |jq .


{


  "agent_reference": {


    "type": "agent_reference",


    "name": "project-planning-agent",


    "version": "7"


  },


  "input": "Gib mir eine Liste aller qwert mit ihrer ID und ihrem Namen. Nutze ask_ontology, um die tatsaechlichen Datensaetze abzufragen, nicht nur Entity-Metadaten. Falls der Begriff unbekannt ist, sage dies, statt eine Bedeutung zu erraten.",


  "store": false


}


## 5. Ohne Synonym mit normaler Endnutzerfrage testen


In [9]:
RES_ORIGINAL=$(az rest --method post --url "$PROJECT_ENDPOINT/openai/v1/responses" \
  --resource https://ai.azure.com --output json --body "$REQUEST" \
  --query "{status:status,error:error,incomplete_details:incomplete_details,
      tools:output[?type=='mcp_list_tools'].{server:server_label,names:tools[].name},
      calls:output[?type=='mcp_call'].{server:server_label,name:name,arguments:arguments,status:status,error:error,result:output},
      answer:output[?type=='message'].content[].text}") &&
printf '%s\n' "$RES_ORIGINAL" | jq . &&
printf '%s\n' "$RES_ORIGINAL" | jq -se -f scripts/check_agent_response.jq > /dev/null


{


  "answer": [


    "Ich kann keine Liste aller \"qwert\" liefern, weil dieses Konzept im aktuellen Ontologie-Modell nicht vorhanden ist. Im Modell sind nur diese Entitäten vorhanden: Email, Project, ProjectDocument, ProjectManager, Task ."


  ],


  "calls": [


    {


... (18 outputs left out)

      "server": "fabric-iq-ontology"


    }


  ]


}


### Gegenprobe ohne Synonym mit erweiterten Hinweisen

Diese Anfrage dient nur der Diagnose. Die normale Frage oben bleibt der
Endnutzertest. `qwert` ist fuer beide Anfragen temporaer entfernt; dies ist nicht der dauerhafte Stand.


In [10]:
RES_WITHOUT_SYNONYM_HINTS=$(az rest --method post --url "$PROJECT_ENDPOINT/openai/v1/responses" \
  --resource https://ai.azure.com --output json --body "$REQUEST_WITH_HINTS" \
  --query "{status:status,error:error,incomplete_details:incomplete_details,
      tools:output[?type=='mcp_list_tools'].{server:server_label,names:tools[].name},
      calls:output[?type=='mcp_call'].{server:server_label,name:name,arguments:arguments,status:status,error:error,result:output},
      answer:output[?type=='message'].content[].text}") &&
printf '%s\n' "$RES_WITHOUT_SYNONYM_HINTS" | jq . &&
printf '%s\n' "$RES_WITHOUT_SYNONYM_HINTS" | jq -se -f scripts/check_agent_response.jq > /dev/null


{


  "answer": [


    "Der Begriff „qwert“ ist in der zugrunde liegenden Ontologie nicht bekannt und es gibt kein entsprechendes modelliertes Objekt oder Synonym. Wenn du möchtest, kann ich dir stattdessen eine Liste einer der folgenden bekannten Entitäten mit ID und Name geben:\n\n- Email\n- Project\n- ProjectDocument\n- ProjectManager\n- Task\n\nGib gerne Bescheid, für welche dieser Entitäten du eine Liste möchtest!【message_1:0†fabric-iq-ontology】"


  ],


  "calls": [


    {


... (18 outputs left out)

      "server": "fabric-iq-ontology"


    }


  ]


}


## 6. Dauerhafte Ontologie mit qwert hochladen

* [upload_ontology_definition.py](../scripts/upload_ontology_definition.py)
* [REST-Endpunkt](https://learn.microsoft.com/rest/api/fabric/ontology/items/update-ontology-definition)

> NOTE: Das [Portal erlaubt die Bearbeitung einzelner Entity-Metadaten](https://learn.microsoft.com/fabric/iq/ontology/how-to-add-metadata).
Das ist von einer oeffentlichen REST-Schnittstelle zu unterscheiden:
Die [dokumentierte Operationsliste](https://learn.microsoft.com/rest/api/fabric/ontology/items)
enthaelt keinen Entity-Metadaten-PATCH.

### Live-Metadaten vor dem Test-Upload


In [11]:
(
  set -euo pipefail
  az rest --method post --url "$ONTOLOGY_URL/getDefinition" \
    --resource https://api.fabric.microsoft.com --body '{}' --output json |
    jq -er '
      def has_qwert: split("\n") | any(rtrimstr("\r") == "\tsynonym qwert");
      [.definition.parts[] | select(.path=="entities/Project.tmdl" and .payloadType=="InlineBase64")]
      | if length==1 then .[0].payload | @base64d
        else error("Genau ein Project-TMDL aus Fabric erwartet") end
      | if (has_qwert | not) then .
        else error("Unerwarteter qwert-Stand in Fabric") end
    ' |
    sed -n 's/\r$//; /^\/\/\//p; /^entity /p; /^\tsynonym /p; /^\tannotation /p'
)


/// A construction project for a customer.


entity Project


	synonym 'Construction project'


	synonym 'Building project'


In [12]:
.venv/bin/python scripts/upload_ontology_definition.py "$WORKSPACE_ID" "$ONTOLOGY_ID" notebooks/fixtures/project_synonym/fabric-ontology-qwert.json

Ontology updated from notebooks/fixtures/project_synonym/fabric-ontology-qwert.json (HTTP 200).


In [13]:
(
  set -euo pipefail
  az rest --method post --url "$ONTOLOGY_URL/getDefinition" \
    --resource https://api.fabric.microsoft.com --body '{}' --output json |
    jq -er '
      def has_qwert: split("\n") | any(rtrimstr("\r") == "\tsynonym qwert");
      [.definition.parts[] | select(.path=="entities/Project.tmdl" and .payloadType=="InlineBase64")]
      | if length==1 then .[0].payload | @base64d
        else error("Genau ein Project-TMDL aus Fabric erwartet") end
      | if has_qwert then .
        else error("Unerwarteter qwert-Stand in Fabric") end
    ' |
    sed -n 's/\r$//; /^\/\/\//p; /^entity /p; /^\tsynonym /p; /^\tannotation /p'
)


/// A construction project for a customer.


entity Project


	synonym 'Construction project'


	synonym 'Building project'


	synonym qwert


## 7. Mit Synonym und normaler Endnutzerfrage testen

Wir senden denselben `REQUEST` wie ohne Synonym, mit derselben Agentenversion
und demselben Ausgabefilter. Es gibt keine zusaetzliche Wartezeit.
Die Tool-Auswahl bleibt automatisch. Vergleiche die tatsaechlichen Tool-Aufrufe,
Ergebnisse und Antworten, nicht nur den Status `completed`.


In [14]:
printf 'Keine zusaetzliche Wartezeit zwischen Upload und Anfrage.\n'


Keine zusaetzliche Wartezeit zwischen Upload und Anfrage.


In [15]:
RES_QWERT=$(az rest --method post --url "$PROJECT_ENDPOINT/openai/v1/responses" \
  --resource https://ai.azure.com --output json --body "$REQUEST" \
  --query "{status:status,error:error,incomplete_details:incomplete_details,
      tools:output[?type=='mcp_list_tools'].{server:server_label,names:tools[].name},
      calls:output[?type=='mcp_call'].{server:server_label,name:name,arguments:arguments,status:status,error:error,result:output},
      answer:output[?type=='message'].content[].text}") &&
printf '%s\n' "$RES_QWERT" | jq . &&
printf '%s\n' "$RES_QWERT" | jq -se -f scripts/check_agent_response.jq > /dev/null


{


  "answer": [


    "Hier ist die Liste aller qwert (interpretiert als \"Project\") mit ihrer ID und ihrem Namen:\n\n| ID    | Name                        |\n|-------|-----------------------------|\n| P-100 | Office Building Hamburg     |\n| P-200 | Residential Complex Munich  |\n\nWenn du möchtest, kann ich die Liste auch sortiert nach Name oder im CSV-Format bereitstellen ."


  ],


  "calls": [


    {


... (18 outputs left out)

      "server": "fabric-iq-ontology"


    }


  ]


}


### Zusaetzliche Diagnose mit Hinweisen und aktivem Synonym

Wir senden denselben `REQUEST_WITH_HINTS` wie bei der Gegenprobe ohne Synonym.
Diese Diagnose ersetzt nicht den Test mit der normalen Endnutzerfrage.
Keine weitere Wartezeit und kein weiterer Upload.


In [16]:
RES_WITH_HINTS=$(az rest --method post --url "$PROJECT_ENDPOINT/openai/v1/responses" \
  --resource https://ai.azure.com --output json --body "$REQUEST_WITH_HINTS" \
  --query "{status:status,error:error,incomplete_details:incomplete_details,
      tools:output[?type=='mcp_list_tools'].{server:server_label,names:tools[].name},
      calls:output[?type=='mcp_call'].{server:server_label,name:name,arguments:arguments,status:status,error:error,result:output},
      answer:output[?type=='message'].content[].text}") &&
printf '%s\n' "$RES_WITH_HINTS" | jq . &&
printf '%s\n' "$RES_WITH_HINTS" | jq -se -f scripts/check_agent_response.jq > /dev/null


{


  "answer": [


    "Hier ist die Liste aller qwert (interpretiert als Projekte) mit ihrer ID und ihrem Namen:\n\n| ID    | Name                        |\n|-------|-----------------------------|\n| P-100 | Office Building Hamburg     |\n| P-200 | Residential Complex Munich  |\n\nDer Begriff \"qwert\" wurde als \"Project\" interpretiert. Falls das nicht gemeint war, bitte den korrekten Begriff angeben, da \"qwert\" nicht explizit im System bekannt ist ."


  ],


  "calls": [


    {


... (18 outputs left out)

      "server": "fabric-iq-ontology"


    }


  ]


}


## 8. Hier wird die Ontology zurueckgesetzt

Diese Zelle laedt `fabric-ontology-repository.json` hoch und stellt damit den
dauerhaften Stand inklusive `synonym qwert` wieder her. qwert wird nicht entfernt.
Immer ausfuehren, auch wenn Abschnitt 7 fehlschlaegt oder uebersprungen wird.


### Live-Metadaten vor dem Ruecksetzen


In [17]:
(
  set -euo pipefail
  az rest --method post --url "$ONTOLOGY_URL/getDefinition" \
    --resource https://api.fabric.microsoft.com --body '{}' --output json |
    jq -er '
      def has_qwert: split("\n") | any(rtrimstr("\r") == "\tsynonym qwert");
      [.definition.parts[] | select(.path=="entities/Project.tmdl" and .payloadType=="InlineBase64")]
      | if length==1 then .[0].payload | @base64d
        else error("Genau ein Project-TMDL aus Fabric erwartet") end
      | if has_qwert then .
        else error("Unerwarteter qwert-Stand in Fabric") end
    ' |
    sed -n 's/\r$//; /^\/\/\//p; /^entity /p; /^\tsynonym /p; /^\tannotation /p'
)


/// A construction project for a customer.


entity Project


	synonym 'Construction project'


	synonym 'Building project'


	synonym qwert


In [18]:
.venv/bin/python scripts/upload_ontology_definition.py "$WORKSPACE_ID" "$ONTOLOGY_ID" notebooks/fixtures/project_synonym/fabric-ontology-repository.json


Ontology updated from notebooks/fixtures/project_synonym/fabric-ontology-repository.json (HTTP 200).


### Live-Metadaten nach dem Ruecksetzen


In [19]:
(
  set -euo pipefail
  az rest --method post --url "$ONTOLOGY_URL/getDefinition" \
    --resource https://api.fabric.microsoft.com --body '{}' --output json |
    jq -er '
      def has_qwert: split("\n") | any(rtrimstr("\r") == "\tsynonym qwert");
      [.definition.parts[] | select(.path=="entities/Project.tmdl" and .payloadType=="InlineBase64")]
      | if length==1 then .[0].payload | @base64d
        else error("Genau ein Project-TMDL aus Fabric erwartet") end
      | if has_qwert then .
        else error("Unerwarteter qwert-Stand in Fabric") end
    ' |
    sed -n 's/\r$//; /^\/\/\//p; /^entity /p; /^\tsynonym /p; /^\tannotation /p'
) &&
jq -ne \
  --argjson baseline "$RES_ORIGINAL" \
  --argjson baseline_hints "$RES_WITHOUT_SYNONYM_HINTS" \
  --argjson positive "$RES_QWERT" \
  --argjson positive_hints "$RES_WITH_HINTS" '
    def ids: [scan("P-[0-9]+")] | unique;
    def check($response; $expected):
      $response.calls[0] as $call
      | $call.name == "ask_ontology"
        and $call.status == "completed"
        and ($call.arguments | fromjson | .request | contains("qwert"))
        and (($response.answer | join("\n") | ids) == $expected)
        and (($call.result | fromjson | .response | ids) == $expected);
    [
      {case:"without_qwert", passed:check($baseline; [])},
      {case:"without_qwert_with_hints", passed:check($baseline_hints; [])},
      {case:"with_qwert", passed:check($positive; ["P-100","P-200"])},
      {case:"with_qwert_with_hints", passed:check($positive_hints; ["P-100","P-200"])}
    ] as $checks
    | if all($checks[]; .passed) then $checks
      else error("Synonym-Testmatrix fehlgeschlagen: " + ($checks | tojson)) end
  '


/// A construction project for a customer.


entity Project


	synonym 'Construction project'


	synonym 'Building project'


	synonym qwert


[


... (13 outputs left out)

    "case": "with_qwert_with_hints",


    "passed": true


  }


]


## Ruecksetzen nach einem Kernel-Neustart

Konfiguration und ID-Abfragen erneut ausfuehren, aber nicht die Testzellen.
Falls die JSON-Dateien fehlen, auch die Dateierzeugung erneut ausfuehren. Danach die Ruecksetzzelle in Abschnitt 8
ausfuehren.

Zurueckgesetzt wird bewusst auf den Repository-Stand, nicht auf einen
zuvor aus Fabric gesicherten Stand. Der Repository-Stand enthaelt qwert dauerhaft. Die vier Antworten stehen nur in
den Notebook-Ausgaben.


## Aussagekraft des Tests

Der Repository-Stand enthaelt `qwert` dauerhaft. Der Durchlauf entfernt es nur
fuer die Negativkontrolle und prueft vier
unabhaengige Anfragen mit derselben gespeicherten Agentenversion:

* Ohne Synonym, normale Endnutzerfrage
* Ohne Synonym, erweiterte Diagnosefrage
* Mit Synonym, dieselbe normale Endnutzerfrage
* Mit Synonym, dieselbe erweiterte Diagnosefrage

Der relevante Endnutzertest ist der Vergleich der beiden normalen Fragen.
Die erweiterten Fragen sind nur Gegenproben. Kein Request erzwingt ein Tool
ueber die API oder schreibt die Zuordnung `qwert` zu Project im Prompt fest.
Die Live-Metadaten pruefen den jeweiligen Synonymstand. Nach dem Wiederherstellen
muss qwert vorhanden sein. Die letzte Codezelle prueft zusaetzlich die vier
Antworten und Tool-Ergebnisse: ohne qwert keine Projekt-IDs, mit qwert genau
P-100 und P-200, jeweils mit ask_ontology als erstem Tool und qwert in der Tool-Frage.
Ein leerer Entity-Metadatenfilter ist kein Beweis fuer fehlende Geschaeftsdaten.

Ein einzelner Durchlauf liefert keine statistische Zuverlaessigkeitsaussage.
Eine erfolgreiche Definition-Aktualisierung garantiert nicht, dass jedes Tool
bereits den neuen Stand verwendet. Moderationswarnungen und widerspruechliche
Antworten sind gesondert zu beurteilen.
